In [ ]:
import os
from pathlib import Path
import datetime 

import numpy as np
import pandas as pd
from astropy.table import Table, join, hstack, vstack
from astropy.coordinates import SkyCoord
import astropy.units as u
import matplotlib.pyplot as plt
from scipy.stats import binned_statistic
import matplotlib.dates as md
from scipy.integrate import trapezoid

from utils import better_step
from matplotlib.gridspec import GridSpec
from statsmodels.stats.proportion import proportion_confint
import seaborn as sns

In [ ]:
# figure defaults for AASTEX AJ

COLUMN_WIDTH = 242.26653/72.27 #in inches
TEXT_WIDTH = 513.11743/72.27
SMALL_SIZE = 9 # in pts
NORMAL_SIZE = 10 
BIG_SIZE = 12
FONT_FAMILY = 'Nimbus Roman No9 L'



params = {
    "font.family": FONT_FAMILY,
    "font.size": NORMAL_SIZE,
    "axes.titlesize": NORMAL_SIZE,
    "axes.labelsize": NORMAL_SIZE,
    
    "xtick.labelsize": SMALL_SIZE,
    "ytick.labelsize": SMALL_SIZE,
    "xtick.top": True,
    "ytick.right": True,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "legend.fontsize": NORMAL_SIZE,
    
    "figure.facecolor": "w",
    "figure.dpi": 300,
    'mathtext.fontset' : "cm"
    
}

plt.rcParams.update(params)

In [ ]:
base_path = Path(os.environ["CFS"]) / "desi" / "users" / "bid13" / "DESI_II"
deep_path = base_path / "other_surveys" / "DEEP23"

In [ ]:
hsc_groth_cat = Table.read(deep_path / "HSC_GROTH.fits").to_pandas()
deep_spec_cat = Table.read(deep_path / "alldeep.egs.2012jun13.fits.gz").to_pandas()
deep_spec_cat = deep_spec_cat[deep_spec_cat["ZQUALITY"]>0]
deep_spec_cat = deep_spec_cat[deep_spec_cat["Z"]>0.001]

zcosmos_spec_cat = Table.read(base_path / "other_surveys" / "ZCOSMOS" / "zcosmos-bright.fits").to_pandas()
zcosmos_spec_cat = zcosmos_spec_cat[zcosmos_spec_cat["REDSHIFT"]>0.001]



cat_desi = Table.read(base_path / "pilot_obs" / "MERGED" / "merged_cat_LSST_WL_Y1.fits" )

cat_desi["success"] = (cat_desi["VI_quality"]>2)
cat_desi = vstack([cat_desi[~cat_desi["success"]],
                      cat_desi[cat_desi["success"] & (cat_desi["Z"]>0.001)]])


In [ ]:
def flux_to_mag(flux):
    return -2.5*np.log10(flux*1e-9) + 8.90

In [ ]:
def process_hsc_cat(hsc_cat):
    hsc_cat["mag_i"] = flux_to_mag(hsc_cat["i_cmodel_flux"])-hsc_cat["a_i"]
    hsc_cat["mag_g"] = flux_to_mag(hsc_cat["g_cmodel_flux"])-hsc_cat["a_g"]
    hsc_cat["mag_r"] = flux_to_mag(hsc_cat["r_cmodel_flux"])-hsc_cat["a_r"]
    hsc_cat["mag_z"] = flux_to_mag(hsc_cat["z_cmodel_flux"])-hsc_cat["a_z"]

    hsc_cat["mag_i_fiber"] = flux_to_mag(hsc_cat["i_fiber_flux"])-hsc_cat["a_i"]
    
    ## Quality cuts
    # valid I-band flux
    qmask = np.isfinite(hsc_cat["i_cmodel_flux"]) & (hsc_cat["i_cmodel_flux"]>0)
    #cmodel fit not failed
    qmask &= (~hsc_cat["i_cmodel_flag"].values)
    #General Failure Flag
    qmask &= (~hsc_cat["i_sdsscentroid_flag"].values)

    # Possible cuts: Bright objects nearby, bad pixels

    #star-galaxy separation (is point source in I band)
    # mask &= (hsc_cat["i_extendedness_value"]>0)


    i_min = 22
    i_max = 24.5
    mask = (hsc_cat["mag_i"] <i_max) & (hsc_cat["mag_i"] >i_min)
    i_fiber_min = 22
    i_fiber_max = 24.75
    mask &= (hsc_cat["mag_i_fiber"] <i_fiber_max) & (hsc_cat["mag_i_fiber"] >i_fiber_min)
    
    return hsc_cat[qmask & mask]

In [ ]:
hsc_groth_cat = process_hsc_cat(hsc_groth_cat)
hsc_coord = SkyCoord(ra=hsc_groth_cat["ra"].values*u.degree, dec=hsc_groth_cat["dec"].values*u.degree)
deep_coord = SkyCoord(ra=deep_spec_cat["RA"].values*u.degree, dec=deep_spec_cat["DEC"].values*u.degree)

idx, d2d, d3d = deep_coord.match_to_catalog_sky(hsc_coord)
phot_cat = hsc_groth_cat.iloc[idx]

idx_mask = d2d.arcsec<1


cat_deep = pd.concat([deep_spec_cat.reset_index(),phot_cat.reset_index()], axis=1)

cat_deep = cat_deep[idx_mask]
print(len(cat_deep)/len(deep_coord))

In [ ]:
hsc_cosmos_cat = process_hsc_cat(Table.read(base_path / "target_data" / "HSC_COSMOS_I_mag_lim_24.8.fits").to_pandas())
hsc_coord = SkyCoord(ra=hsc_cosmos_cat["ra"].values*u.degree, dec=hsc_cosmos_cat["dec"].values*u.degree)
zcosmos_coord = SkyCoord(ra=zcosmos_spec_cat["RAJ2000"].values*u.degree, dec=zcosmos_spec_cat["DEJ2000"].values*u.degree)

idx, d2d, d3d = zcosmos_coord.match_to_catalog_sky(hsc_coord)
phot_cat = hsc_cosmos_cat.iloc[idx]

idx_mask = d2d.arcsec<1


cat_zcosmos = pd.concat([zcosmos_spec_cat.reset_index(),phot_cat.reset_index()], axis=1)

cat_zcosmos = cat_zcosmos[idx_mask]
print(len(cat_zcosmos)/len(zcosmos_coord))

In [ ]:
cat_deep["success"] = cat_deep["ZQUALITY"]>2
###Figure out how to merge VI spectypes
cat_deep = cat_deep[cat_deep["Z"]>0.001]
cat_zcosmos["success"] = np.isin(cat_zcosmos["CC"].astype(int), [3,4,13,14,23,24,213,214])
cat_zcosmos = cat_zcosmos[cat_zcosmos["REDSHIFT"]>0.001]

In [ ]:
def make_success_plot(cat,colname="mag_i", nbins=10, ref_sample=None, ax =None,range=None, **kwargs):
    success, bin_edges , _ = binned_statistic(cat[colname], cat["success"],bins=nbins,statistic="mean",range=range)
    sums, bin_edges , _ = binned_statistic(cat[colname], cat["success"],bins=nbins,statistic="sum",range=range)
    count, bin_edges, _ = binned_statistic(cat[colname], cat["success"],bins=nbins,statistic="count",range=range)
    # print(sums)
    # print(count)
    adjustment =1.0
    if ref_sample is not None:
        for i in range(len(bin_edges)-1):
            sel_cat = ref_sample[(ref_sample["HSC_i_MAG"]>=bin_edges[i]) & (ref_sample["HSC_i_MAG"]<bin_edges[i])]
            frac_greater = np.sum(sel_cat["Z"]>1.6)/len(sel_cat)
            print(frac_greater)
    # e_success = np.sqrt(success*(1-success)/(count*adjustment))
    # e_success = 1.96 * e_success # 95% CI
    ci_lo, ci_upp = proportion_confint(sums,count,method="beta")
    
    ax = better_step(bin_edges, success, yerr=(ci_lo,ci_upp), ax = ax,**kwargs)
    # print(success)
    # ax.set_ylim(0.3,1)
    # ax.grid(linestyle="--")
    return ax

### Plots With(out) Hercules

In [ ]:
with_hercules = True
cat_use = cat_desi.copy()

if  not with_hercules:
    cat_desi = cat_use[cat_use["FIELD_NAME"]!="HERCULES"]

In [ ]:
time_bins=[(70,100),(100,143),(143,204),(204,292),(292,417)]

mag_bin_edges = [22,22.5,23,23.75, 24.5]


fig = plt.figure( layout="tight",figsize=(TEXT_WIDTH,0.6*TEXT_WIDTH))
gs = GridSpec(2, 6, figure=fig)
axs = [fig.add_subplot(gs[0, :2]), fig.add_subplot(gs[0, 2:4]), fig.add_subplot(gs[0, 4:6]),
      fig.add_subplot(gs[1, 1:3]), fig.add_subplot(gs[1, 3:5])]

cat_zcosmos = cat_zcosmos[cat_zcosmos["mag_i"]<=22.5]
zcosmos_success = cat_zcosmos["success"].mean()
ci_lo_zcosmos, ci_upp_zcosmos = proportion_confint(cat_zcosmos["success"].sum(),len(cat_zcosmos),method="beta")

for i, t in enumerate(time_bins):
    cat_plot = cat_desi[(cat_desi["EXPTIME"]/60>=t[0]) & (cat_desi["EXPTIME"]/60<t[1])]
    make_success_plot(cat_plot, ax= axs[i],label="DESI")
    # if i == 1:
    make_success_plot(cat_deep, ax= axs[i], label="DEEP-2/3",ls="--",alpha=0.8)
    # axs[i].errorbar(22.25,zcosmos_success,xerr=np.array([0.25,0.25])[:,None], color="gray", label="zCOSMOS-Bright")
    make_success_plot(cat_zcosmos,nbins=1, ax= axs[i],label="zCOSMOS-Bright",color="gray")
    # axs[i].scatter(22.25,zcosmos_success,marker="o",c="gray",,)
    # make_success_plot(cat_zcosmos, ax= axs[i],label="ZCOSMOS-Bright", ls="--")
    axs[i].set_title(f"{t[0]} $\leq$"+"exp. time [min]"+f"$<${t[1]}")
    axs[i].set_ylim(0,1.04)
    axs[i].set_xlim(21.9,24.6)
    axs[i].set_xlabel("$i$-magnitude")
axs[0].set_ylabel("Success Rate")
axs[3].set_ylabel("Success Rate")
axs[1].legend(frameon=False,borderaxespad=0.5,borderpad=0.,fontsize=SMALL_SIZE,handlelength=1.5)


if with_hercules:
    plt.savefig("./figs/i_mag_success_with_hercules.pdf",bbox_inches="tight",dpi=300)
else:
    plt.savefig("./figs/i_mag_success.pdf",bbox_inches="tight",dpi=300)

In [ ]:
time_bins=[(70,100),(100,143),(143,204),(204,292),(292,417)]

mag_bin_edges = [22,22.5,23,23.75, 24.5]


fig = plt.figure( layout="tight",figsize=(TEXT_WIDTH,0.6*TEXT_WIDTH))
gs = GridSpec(2, 6, figure=fig)
axs = [fig.add_subplot(gs[0, :2]), fig.add_subplot(gs[0, 2:4]), fig.add_subplot(gs[0, 4:6]),
      fig.add_subplot(gs[1, 1:3]), fig.add_subplot(gs[1, 3:5])]


for i, t in enumerate(time_bins):
    cat_plot = cat_desi[(cat_desi["EXPTIME"]/60>=t[0]) & (cat_desi["EXPTIME"]/60<t[1])]
    make_success_plot(cat_plot, ax= axs[i],label="DESI", colname="mag_i_fiber")
    # if i == 1:
    make_success_plot(cat_deep, ax= axs[i], label="DEEP-2/3",ls="--",alpha=0.8,colname="mag_i_fiber")
    make_success_plot(cat_zcosmos,nbins=1, ax= axs[i],label="zCOSMOS-Bright",color="gray")
    # make_success_plot(cat_zcosmos, ax= axs[i],label="ZCOSMOS-Bright", ls="--")
    axs[i].set_title(f"{t[0]} $\leq$"+"Exp. Time [min]"+f"$<${t[1]}")
    axs[i].set_ylim(0,1.04)
    axs[i].set_xlim(21.9,25.1)
    axs[i].set_xlabel("$i$-fiber-magnitude")
axs[0].set_ylabel("Success Rate")
axs[3].set_ylabel("Success Rate")
axs[1].legend(frameon=False,borderaxespad=0.5,borderpad=0.,fontsize=SMALL_SIZE,handlelength=1.5)


if with_hercules:
    plt.savefig("./figs/i_fiber_mag_success_with_hercules.pdf",bbox_inches="tight",dpi=300)
else:
    plt.savefig("./figs/i_fiber_mag_success.pdf",bbox_inches="tight",dpi=300)

### Success as a function of time in bins of magnitude

In [ ]:
time_bins=[(70,100),(100,143),(143,204),(204,292),(292,417)]

mag_bin_edges = [(22,22.5),(22.5,23),(23,23.5),(23.5,24),(24,24.5),(24.5,25)]


fig = plt.figure( layout="tight",figsize=(TEXT_WIDTH,0.6*TEXT_WIDTH))
gs = GridSpec(2, 6, figure=fig)
axs = [fig.add_subplot(gs[0, :2]), fig.add_subplot(gs[0, 2:4]), fig.add_subplot(gs[0, 4:6]),
      fig.add_subplot(gs[1, 1:3]), fig.add_subplot(gs[1, 3:5])]


for i, t in enumerate(mag_bin_edges[:-1]):
    
    cat_plot = cat_desi[(cat_desi["mag_i"]>=t[0]) & (cat_desi["mag_i"]<t[1])]
    cat_plot["EXPTIME"] = cat_plot["EXPTIME"]/60
    make_success_plot(cat_plot, ax= axs[i],label="DESI", colname="EXPTIME",range=(0,420))
    # if i == 1:
    # make_success_plot(cat_deep, ax= axs[i], label="DEEP-2/3",ls="--",alpha=0.8,colname="mag_i_fiber")
    # make_success_plot(cat_zcosmos, ax= axs[i],label="ZCOSMOS-Bright", ls="--")
    axs[i].set_title(f"{t[0]} $\leq$"+"$i$-mag"+f"$<${t[1]}")
    axs[i].set_ylim(0,1.04)
    axs[i].set_xticks([15,100,200,300,400])
    axs[i].set_xlim(-10,450)
    axs[i].set_xlabel("Exposure Time [min]")
axs[0].set_ylabel("Success Rate")
axs[3].set_ylabel("Success Rate")
# axs[1].legend(frameon=False)

if with_hercules:
    plt.savefig("./figs/exp_time_success_with_hercules.pdf",bbox_inches="tight",dpi=300)
else:
    plt.savefig("./figs/exp_time_success.pdf",bbox_inches="tight",dpi=300)

In [ ]:
time_bins=[(70,100),(100,143),(143,204),(204,292),(292,417)]

mag_bin_edges = [(22,22.5),(22.5,23),(23,23.5),(23.5,24),(24,24.5),(24.5,25)]


fig,axs = plt.subplots(2,3,layout="tight",figsize=(TEXT_WIDTH,0.6*TEXT_WIDTH))
axs=np.ravel(axs)


for i, t in enumerate(mag_bin_edges):
    
    cat_plot = cat_desi[(cat_desi["mag_i_fiber"]>=t[0]) & (cat_desi["mag_i_fiber"]<t[1])]
    cat_plot["EXPTIME"] = cat_plot["EXPTIME"]/60
    make_success_plot(cat_plot, ax= axs[i],label="DESI", colname="EXPTIME",range=(0,420))
    # if i == 1:
    # make_success_plot(cat_deep, ax= axs[i], label="DEEP-2/3",ls="--",alpha=0.8,colname="mag_i_fiber")
    # make_success_plot(cat_zcosmos, ax= axs[i],label="ZCOSMOS-Bright", ls="--")
    axs[i].set_title(f"{t[0]} $\leq$"+"$i$-fiber-mag"+f"$<${t[1]}")
    axs[i].set_xticks([15,100,200,300,400])
    axs[i].set_ylim(0,1.04)
    axs[i].set_xlim(-10,450)
    axs[i].set_xlabel("Exposure Time [min]")
axs[0].set_ylabel("Success Rate")
axs[3].set_ylabel("Success Rate")
# axs[1].legend(frameon=False)


if with_hercules:
    plt.savefig("./figs/exp_time_success_i_fiber_with_hercules.pdf",bbox_inches="tight",dpi=300)
else:
    plt.savefig("./figs/exp_time_success_i_fiber.pdf",bbox_inches="tight",dpi=300)

In [ ]:

# ax = make_success_plot(cat_desi)
# ax = make_success_plot(cat_deep)
# ax = make_success_plot(cat_zcosmos, ax=ax)
# ax.set_title("All Objects")
# ax.set_xlabel("$i$-mag")

# ax = make_success_plot(cat_desi, colname="mag_i_fiber")
# ax = make_success_plot(cat_deep, colname="mag_i_fiber")
# ax = make_success_plot(cat_zcosmos, ax=ax, colname="mag_i_fiber")
# ax.set_title("All Objects")
# ax.set_xlabel("$i$-fiber-mag")

In [ ]:
# fig, axs = plt.subplots(1,1, figsize=(COLUMN_WIDTH,0.8*COLUMN_WIDTH))
# # cat_sel = cat_desi[(cat_desi["EXPTIME"]>100*60) & (cat_desi["EXPTIME"]<=143*60)]
# cat_sel = cat_desi_wh[(cat_desi_wh["EXPTIME"]/60>=100) & (cat_desi_wh["EXPTIME"]/60<143)]

# make_success_plot(cat_sel, ax= axs,label="DESI")
# make_success_plot(cat_deep, ax= axs, ls="--", label="DEEP-2/3")
# make_success_plot(cat_zcosmos[cat_zcosmos["mag_i"]<22.5], ax= axs, ls="--", label="zCOSMOS",nbins=2)
# axs.set_ylabel("Success Rate")
# axs.set_xlabel("$i$-mag")
# axs.set_ylim(0.1,1)
# axs.legend(loc="lower left",frameon=False)
# # plt.savefig("p5-desi-faint.png",dpi=300, bbox_inches='tight')

### Treat z>1.45 as incorrect

In [ ]:
# names = [name for name in cat_desi.colnames if len(cat_desi[name].shape) <= 1]
# cat_desi_short = cat_desi[names].to_pandas()
# # cat_desi_short = cat_desi.copy()
# cat_desi_short.loc[cat_desi_short["Z"]>1.45,"success"] = False
# mag_bin_edges = [22,22.5,23,23.75, 24.5]
# fig, axs = plt.subplots(1,3, figsize=(17,4.5))
# axs = np.ravel(axs)

# # cat_list = [cat_desi, cat_deep, cat_zcosmos]
# # for cat in cat_list:
# cat_sel = cat_desi_short[(cat_desi_short["EXPTIME"]<=1.25*3600)]
# make_success_plot(cat_sel, ax= axs[0],label="DESI")
# make_success_plot(cat_deep, ax= axs[0], label="DEEP-2/3", ls="--")
# make_success_plot(cat_zcosmos, ax= axs[0],label="ZCOSMOS-Bright", ls="--")
# axs[0].set_title("25m to 1h15m", fontsize=20)
# axs[0].legend(frameon=False)
# # print(cat_sel["EXPTIME"].min(),cat_sel["EXPTIME"].max())

# cat_sel = cat_desi_short[(cat_desi_short["EXPTIME"]>1.25*60) & (cat_desi_short["EXPTIME"]<=3.75*3600)]
# make_success_plot(cat_sel, ax= axs[1])
# make_success_plot(cat_deep, ax= axs[1], ls="--")
# make_success_plot(cat_zcosmos, ax= axs[1], ls="--")
# axs[1].set_title("1h15m to 3h45m", fontsize=20)
# # print(cat_sel["EXPTIME"].min(),cat_sel["EXPTIME"].max())

# cat_sel = cat_desi_short[(cat_desi_short["EXPTIME"]>3.75*3600)]
# make_success_plot(cat_sel, ax= axs[2])
# make_success_plot(cat_deep, ax= axs[2], ls="--")
# make_success_plot(cat_zcosmos, ax= axs[2], ls="--")
# axs[2].set_title("3h45m to 8h", fontsize=20)
# # print(cat_sel["EXPTIME"].min(),cat_sel["EXPTIME"].max())

# axs[0].set_ylabel("Success Rate", fontsize=25)
# axs[1].set_xlabel("$i$-mag", fontsize=25)
# # fig.suptitle("*considering $z>1.45$ as a failure", y=1.05, fontsize=15)
# plt.savefig("i_mag_success_DEEP_criteria.pdf", bbox_inches='tight')

In [ ]:
# mag_bin_edges = [22,22.5,23,23.75, 24.5]
# fig, axs = plt.subplots(1,3, figsize=(17,4.5))
# axs = np.ravel(axs)

# # cat_list = [cat_desi, cat_deep, cat_zcosmos]
# # for cat in cat_list:
# cat_sel = cat_desi_short[(cat_desi_short["EXPTIME"]<=1.25*3600)]
# make_success_plot(cat_sel, ax= axs[0],label="DESI", colname="mag_i_fiber")
# make_success_plot(cat_deep, ax= axs[0], label="DEEP-2/3", colname="mag_i_fiber", ls="--")
# make_success_plot(cat_zcosmos, ax= axs[0],label="ZCOSMOS-Bright", colname="mag_i_fiber", ls="--")
# axs[0].set_title("25m to 1h15m", fontsize=20)
# axs[0].legend(frameon=False)
# # print(cat_sel["EXPTIME"].min(),cat_sel["EXPTIME"].max())

# cat_sel = cat_desi_short[(cat_desi_short["EXPTIME"]>1.25*60) & (cat_desi_short["EXPTIME"]<=3.75*3600)]
# make_success_plot(cat_sel, ax= axs[1], colname="mag_i_fiber")
# make_success_plot(cat_deep, ax= axs[1], colname="mag_i_fiber", ls="--")
# make_success_plot(cat_zcosmos, ax= axs[1], colname="mag_i_fiber", ls="--")
# axs[1].set_title("1h15m to 3h45m", fontsize=20)
# # print(cat_sel["EXPTIME"].min(),cat_sel["EXPTIME"].max())

# cat_sel = cat_desi_short[(cat_desi_short["EXPTIME"]>3.75*3600)]
# make_success_plot(cat_sel, ax= axs[2], colname="mag_i_fiber")
# make_success_plot(cat_deep, ax= axs[2], colname="mag_i_fiber", ls="--")
# make_success_plot(cat_zcosmos, ax= axs[2], colname="mag_i_fiber", ls="--")
# axs[2].set_title("3h45m to 8h", fontsize=20)
# # print(cat_sel["EXPTIME"].min(),cat_sel["EXPTIME"].max())

# axs[0].set_ylabel("Success Rate", fontsize=25)
# axs[1].set_xlabel("$i$-fiber-mag", fontsize=25)
# # fig.suptitle("*considering $z>1.45$ as a failure", y=1.05, fontsize=15)
# plt.savefig("i_fiber_mag_success_DEEP_criteria.pdf", bbox_inches='tight')

In [ ]:
# # mag_bin_edges = [22,22.5,23,23.75, 24.5]
# fig, axs = plt.subplots(1,2, figsize=(13,4.5))
# axs = np.ravel(axs)

# # cat_list = [cat_desi, cat_deep, cat_zcosmos]
# # for cat in cat_list:
# cat_sel = cat_desi[(cat_desi["mag_i"]<=23)]
# cat_sel = cat_desi[(cat_desi["EXPTIME"]<=20000)]
# # cat_sel["EXPTIME"] = pd.to_datetime(cat_sel["EXPTIME"],unit="s")
# make_success_plot(cat_sel, ax= axs[0],colname="EXPTIME")
# # make_success_plot(cat_deep, ax= axs[0], label="DEEP-2/3", ls="--")
# # make_success_plot(cat_zcosmos, ax= axs[0],label="ZCOSMOS-Bright", ls="--")
# axs[0].set_title("22<$i$-mag$\leq$23", fontsize=20)
# # axs[0].legend(frameon=False)
# axs[0].set_ylim(0.38,1)
# # print(cat_sel["EXPTIME"].min(),cat_sel["EXPTIME"].max())
# axs[0].set_xticks([3600*i for i in range(1,6)], labels=[f"{i}h" for i in range(1,6)])
           
# cat_sel = cat_desi[cat_desi["mag_i"]>23]
# make_success_plot(cat_sel, ax= axs[1],colname="EXPTIME")
# # make_success_plot(cat_deep, ax= axs[1], ls="--")
# # make_success_plot(cat_zcosmos, ax= axs[1], ls="--")
# axs[1].set_title("23<$i$-mag$\leq$24.5", fontsize=20)
# axs[1].set_ylim(0.38,1)
# # print(cat_sel["EXPTIME"].min(),cat_sel["EXPTIME"].max())
# axs[1].set_xticks([3600*i for i in range(1,9)], labels=[f"{i}h" for i in range(1,9)])

# # for i in range(2):
# #     axs[i].xaxis.set_major_formatter(myFormatter)
# #     axs[i].xaxis.set_minor_formatter(myFormatter)
# #     axs[i].yaxis.set_major_formatter(myFormatter)
# #     axs[i].yaxis.set_minor_formatter(myFormatter)

# axs[0].set_ylabel("Success Rate", fontsize=25)
# fig.text(0.35,0.0,"Exposure Time", fontsize=25)
# plt.savefig("exptime_success_imag.pdf", bbox_inches='tight')

In [ ]:
# # mag_bin_edges = [22,22.5,23,23.75, 24.5]
# fig, axs = plt.subplots(1,2, figsize=(13,4.5))
# axs = np.ravel(axs)

# # cat_list = [cat_desi, cat_deep, cat_zcosmos]
# # for cat in cat_list:
# cat_sel = cat_desi[(cat_desi["mag_i_fiber"]<=23)]
# cat_sel = cat_desi[(cat_desi["EXPTIME"]<=20000)]
# # cat_sel["EXPTIME"] = pd.to_datetime(cat_sel["EXPTIME"],unit="s")
# make_success_plot(cat_sel, ax= axs[0],colname="EXPTIME")
# # make_success_plot(cat_deep, ax= axs[0], label="DEEP-2/3", ls="--")
# # make_success_plot(cat_zcosmos, ax= axs[0],label="ZCOSMOS-Bright", ls="--")
# axs[0].set_title("22<$i$-mag-fiber$\leq$23", fontsize=20)
# axs[0].legend(frameon=False)
# axs[0].set_ylim(0.38,1)
# # print(cat_sel["EXPTIME"].min(),cat_sel["EXPTIME"].max())
# axs[0].set_xticks([3600*i for i in range(1,6)], labels=[f"{i}h" for i in range(1,6)])

# cat_sel = cat_desi[cat_desi["mag_i_fiber"]>23]
# make_success_plot(cat_sel, ax= axs[1],colname="EXPTIME")
# # make_success_plot(cat_deep, ax= axs[1], ls="--")
# # make_success_plot(cat_zcosmos, ax= axs[1], ls="--")
# axs[1].set_title("23<$i$-mag-fiber$\leq$24.5", fontsize=20)
# axs[1].set_ylim(0.38,1)
# # print(cat_sel["EXPTIME"].min(),cat_sel["EXPTIME"].max())
# axs[1].set_xticks([3600*i for i in range(1,9)], labels=[f"{i}h" for i in range(1,9)])


# axs[0].set_ylabel("Success Rate", fontsize=25)
# fig.text(0.35,0.0,"Exposure Time", fontsize=25)
# plt.savefig("exptime_success_i_fiber_mag.pdf", bbox_inches='tight')

### Quality Indicator

In [ ]:
rng = np.random.default_rng(42)
fig,ax = plt.subplots(1,1, figsize=(COLUMN_WIDTH,0.7*COLUMN_WIDTH))
noise = rng.uniform(-0.02,0.02, len(cat_desi))
ax.scatter(cat_desi["VI_quality"]+noise,cat_desi["DELTACHI2"], alpha=0.1,facecolor="none",edgecolor="C0",rasterized=True)
ax.set_yscale("log")
ax.set_xlim(-0.3,4.3)
ax.set_ylim(ymax=1e6)
ax.fill_between(np.linspace(3,4.3,10),0,1e6, alpha=0.05,color="k")
ax.set_xlabel("VI Quality Flag")
ax.set_ylabel(r"$\Delta \chi^{2}$")

if with_hercules:
    plt.savefig("./figs/VI_V_deltaCHI2_with_hercules.pdf",bbox_inches="tight",dpi=300)
else:
    plt.savefig("./figs/VI_V_deltaCHI2.pdf",bbox_inches="tight",dpi=300)

# Make corrections for z>1.6 pop

In [ ]:
def z0_for_limit(maglimit, band='i'):
    if band=='i':
        # z0 = -0.724785  +  0.0408182 * maglimit #21.5 < i < 23 # numbers from Jeff's email
        z0 = -0.744  +  0.0417 * maglimit #Number from DESC SRD 
    elif band=='r':
        z0 = -1.10356 +    0.0567708 * maglimit #for 22 < r < 24
    return z0

In [ ]:
def dNdz(z, z0, alpha ):
    return z**2 * np.exp((-z/z0)**alpha)/(2*z0**3)

In [ ]:
def dNdz_cumulative_mag(z_grid,maglimit,alpha=1,band='i'):
    z0 = z0_for_limit(maglimit, band=band)
    return dNdz(z_grid, z0, alpha=alpha)

In [ ]:
z_grid = np.linspace(0,3,100)

for z in [22,23,24.1,25.3]:
    number_den = dNdz_cumulative_mag(z_grid,z,alpha=1,band='i')
    
    plt.plot(z_grid, number_den, label=f"{z}")
plt.axvline(1.6, color="k", ls ="--")
plt.legend(title=r"$i$-mag",title_fontsize=20, frameon=False)
plt.xlabel(r"$z$", fontsize=20)
plt.ylabel(r"$\frac{1}{N_{total}}\dfrac{dn}{dz}$", fontsize=20)

In [ ]:
spec_cat = Table.read( base_path / "other_surveys" / "COSMOS2020" / "COSMOS2020_FARMER_R1_v2.2_p3.fits").to_pandas()
mask = (spec_cat['FLAG_COMBINED']==0) & ((spec_cat['lp_type']==0) | (spec_cat['lp_type']==2))
spec_cat = spec_cat[mask]
z_list = np.zeros(len(spec_cat))
z_list[(spec_cat['lp_type']==0).to_numpy(dtype=bool)] = spec_cat["lp_zBEST"][(spec_cat['lp_type']==0)]
z_list[(spec_cat['lp_type']==2).to_numpy(dtype=bool)] = spec_cat["lp_zq"][(spec_cat['lp_type']==2)]
spec_cat["Z"] = z_list



In [ ]:
i_grid = [22,23,24.1,25.3]
for idx, i_mag in enumerate(i_grid):
    mask = spec_cat["HSC_i_MAG"]<=i_mag
    sel_cat = spec_cat[mask]
    
    z0 = z0_for_limit(i_mag)
    number_den = dNdz(z_grid, z0, 1)
    number_den /=trapezoid(number_den, z_grid)
    
    plt.plot(z_grid, number_den, color=f"C{idx+1}",ls="--")
    
    plt.hist(sel_cat["Z"], bins=50, histtype="step",lw=2,label=f"{i_mag}", density=True, color=f"C{idx+1}")
plt.legend()
plt.xlim(0,3)

In [ ]:
def N_total(i_lim):
    # return 42.9*10**(0.359*(i_lim-25)) # Number from DESC SRD
    return 46*10**(0.31*(i_lim-25)) #Number from LSST Science Book

In [ ]:
def make_success_plot(cat,colname="mag_i", nbins=10, ref_sample=None,analytic_correction=False,plot_error=False, ax =None,bin_range=None, **kwargs):
    success, bin_edges , _ = binned_statistic(cat[colname], cat["success"],bins=nbins,statistic="mean",range=bin_range)
    sums, bin_edges , _ = binned_statistic(cat[colname], cat["success"],bins=nbins,statistic="sum",range=bin_range)
    count, bin_edges, _ = binned_statistic(cat[colname], cat["success"],bins=nbins,statistic="count",range=bin_range)
    # print(sums)
    # print(count)
    threshold=1.6
    frac_greater_list = []
    
    if analytic_correction or ref_sample is not None:
        if analytic_correction:
            for i in range(len(bin_edges)-1):
                z_grid = np.linspace(threshold,100,5000)
                numerator = N_total(bin_edges[i+1])*trapezoid(dNdz_cumulative_mag(z_grid,bin_edges[i+1]),z_grid) - N_total(bin_edges[i])*trapezoid(dNdz_cumulative_mag(z_grid,bin_edges[i]),z_grid)
                denominator = N_total(bin_edges[i+1])*trapezoid(dNdz_cumulative_mag(np.linspace(0,100,5000),bin_edges[i+1]),np.linspace(0,100,5000)) - N_total(bin_edges[i])*trapezoid(dNdz_cumulative_mag(np.linspace(0,100,5000),bin_edges[i]),np.linspace(0,100,5000))
                frac_greater_list.append(numerator/denominator)
            # print(frac_greater_list)

        
        if ref_sample is not None:
            for i in range(len(bin_edges)-1):
                sel_cat = ref_sample[(ref_sample["HSC_i_MAG"]>=bin_edges[i]) & (ref_sample["HSC_i_MAG"]<bin_edges[i+1])]
                frac_greater_list.append( np.sum(sel_cat["Z"]>threshold)/len(sel_cat) )
        # print(frac_greater_list,"\n")
        success, bin_edges , _ = binned_statistic(cat[colname], cat["success"],bins=nbins,statistic="mean")
        sums, bin_edges , _ = binned_statistic(cat[colname], cat["success"],bins=nbins,statistic="sum",range=bin_range)
        count, bin_edges, _ = binned_statistic(cat[colname], cat["success"],bins=nbins,statistic="count",range=bin_range)

        fails = count-sums
        numbers_greater_than = np.array(frac_greater_list) * count
        adjusted_fails = fails - numbers_greater_than
        success = 1 - (adjusted_fails/((1-np.array(frac_greater_list))*count))

        
        # failure = 1-success
        # failure = failure - np.array(frac_greater_list)
        # success = 1-failure
# #         e_success = np.sqrt(success*(1-success)/(count))
        
        
        
#     ci_lo, ci_upp = proportion_confint(sums,count,method="beta")
    ci_lo, ci_upp = proportion_confint(sums,count,method="beta")
    if plot_error:
        yerr=(ci_lo,ci_upp)
    else:
        yerr=None
    ax = better_step(bin_edges, success, yerr=yerr, ax = ax,**kwargs)
    # ax.set_ylim(0.3,1)
    # ax.grid(linestyle="--")
    return ax


In [ ]:
time_bins=[(70,100),(100,143),(143,204),(204,292),(292,417)]

mag_bin_edges = [(22,22.5),(22.5,23),(23,23.5),(23.5,24),(24,24.5),(24.5,25)]



fig = plt.figure( layout="tight",figsize=(TEXT_WIDTH,0.6*TEXT_WIDTH))
gs = GridSpec(2, 6, figure=fig)
axs = [fig.add_subplot(gs[0, :2]), fig.add_subplot(gs[0, 2:4]), fig.add_subplot(gs[0, 4:6]),
      fig.add_subplot(gs[1, 1:3]), fig.add_subplot(gs[1, 3:5])]


for i, t in enumerate(time_bins):
    cat_plot = cat_desi[(cat_desi["EXPTIME"]/60>=t[0]) & (cat_desi["EXPTIME"]/60<t[1])]
    make_success_plot(cat_plot, ax= axs[i],label="Measured", plot_error=True)
    make_success_plot(cat_plot, ax= axs[i],label="COSMOS adjustment",ref_sample=spec_cat, color="C1", alpha=1, ls="--")
    make_success_plot(cat_plot, ax= axs[i],label="Analytic adjustment",analytic_correction=True, alpha=1, color="C2",ls="--")
    axs[i].set_title(f"{t[0]} $\leq$"+"Exp. Time [min]"+f"$<${t[1]}")
    axs[i].set_ylim(0,1.04)
    axs[i].set_xlim(21.9,24.6)
    axs[i].set_xlabel("$i$-magnitude")
axs[0].set_ylabel("Success Rate")
axs[3].set_ylabel("Success Rate")
axs[3].legend(frameon=False,handlelength=1.5)




if with_hercules:
    plt.savefig("./figs/adjusted_success_with_hercules.pdf",bbox_inches="tight",dpi=300)
else:
    plt.savefig("./figs/adjusted_success.pdf",bbox_inches="tight",dpi=300)

# Compare with Deep 2/3

In [ ]:
cat_deep["g-r"] = cat_deep["mag_g"] - cat_deep["mag_r"]
cat_deep["r-z"] = cat_deep["mag_r"] - cat_deep["mag_z"]
cat_desi["g-r"] = cat_desi["mag_g"] - cat_desi["mag_r"]
cat_desi["r-z"] = cat_desi["mag_r"] - cat_desi["mag_z"]

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(TEXT_WIDTH, 0.5*TEXT_WIDTH))

cat_deep2 = cat_deep.sample(len(cat_desi))
ax[0].scatter(cat_deep2["r-z"][cat_deep2["success"]], cat_deep2["g-r"][cat_deep2["success"]],marker=".",s=1,alpha=0.1)
ax[0].scatter(cat_deep2["r-z"][~cat_deep2["success"]], cat_deep2["g-r"][~cat_deep2["success"]],marker=".",s=1)
ax[0].set_xlim(-2,3)
ax[0].set_ylim(3,-1)

ax[1].scatter(cat_desi["r-z"][cat_desi["success"]], cat_desi["g-r"][cat_desi["success"]],marker=".",s=1,alpha=0.1)
ax[1].scatter(cat_desi["r-z"][~cat_desi["success"]], cat_desi["g-r"][~cat_desi["success"]],marker=".",s=1)
ax[1].set_xlim(-2,3)
ax[1].set_ylim(3,-1)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(TEXT_WIDTH, 0.5*TEXT_WIDTH))
c = ax[0].scatter(cat_desi["r-z"][cat_desi["success"]], cat_desi["g-r"][cat_desi["success"]],marker=".",s=1,c=cat_desi["Z"][cat_desi["success"]],vmin=0.01,vmax=1.6)
fig.colorbar(c,)
# ax[0].scatter(cat_desi["r-z"][~cat_desi["success"]], cat_desi["g-r"][~cat_desi["success"]],marker=".",s=1)
ax[0].set_xlim(-2,3)
ax[0].set_ylim(3,-1)

c = ax[1].scatter(cat_desi["r-z"][cat_desi["success"]], cat_desi["g-r"][cat_desi["success"]],marker=".",s=1,c=cat_desi["mag_i"][cat_desi["success"]],vmin=22.5,vmax=24.5)
fig.colorbar(c,)
# ax[1].scatter(cat_desi["r-z"][~cat_desi["success"]], cat_desi["g-r"][~cat_desi["success"]],marker=".",s=1)
ax[1].set_xlim(-2,3)
ax[1].set_ylim(3,-1)